# Notebook 18 — System I: SBI hyperparameter ablation study

## Purpose

A reviewer of a closely related paper by the same group (Casolo et al., accepted at
*Engineering Applications of Artificial Intelligence*) asked, almost verbatim: *"NSF
depth, training set size, and optimizer settings are fixed without justification.
Please vary at least the training set size (10k-100k) and NSF depth or justify."*
This notebook runs that ablation for System I: training budget, network
architecture (width/depth), and density-estimator family, all evaluated against
one shared held-out simulation-based-calibration (SBC) set so every config is
compared on equal footing.

**Efficiency design** (read before rerunning): the expensive part of any System I
data point is the per-draw matched scenario-specific warm-start solve
(~0.35s/draw extra). Generating a fresh bank per ablation config would cost
`n_configs` times the production training cost. Instead:
1. Generate **one** 10,000-draw matched-protocol training bank up front (~85-100 min
   -- this is the one long step, comparable to the existing production posterior's
   own training cost).
2. Generate **one** held-out matched-protocol SBC evaluation set (300 draws, ~3 min).
3. Every ablation config trains on a **prefix** of the shared bank (for the budget
   axis) or the **full** bank (for the architecture/estimator axes) and is scored
   against the **same** held-out set -- each of these retrains costs only
   minutes, since no new simulations are run.

## Ablation axes

| Axis | Values tested | Fixed elsewhere |
|---|---|---|
| Training budget `n_simulations` | 500, 1,000, 2,500, 5,000, 10,000 | NSF, 128 hidden, 5 transforms |
| Architecture (hidden features, transforms) | (64,3), (128,5) [production], (256,5), (128,8) | NSF, n=10,000 |
| Density estimator family | NSF, MAF | 128 hidden, 5 transforms, n=10,000 |

(128, 5, NSF, n=10,000) is the production configuration and appears once, shared
across all three axes as the reference point.

## Outputs

- `data/sysI_ablation_train_bank_n10000.npz` (shared training bank)
- `results/sysI_ablation_sbc_heldout_n300.npz` (shared held-out SBC set)
- `results/sysI_ablation_study.csv` (the full results table)
- `figures/18_ablation_study.png`


## Section 1 — Imports and setup


In [1]:
import time
import pickle
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from cstr_sbi.physics import NOMINAL_INLET_CL, NOMINAL_CTRL, NOMINAL_PARAMS_CL, UA_NOMINAL, K0_NOMINAL
from cstr_sbi.simulator import warm_start_ic, DEFAULT_SENSOR_NOISE_PCT
from cstr_sbi.inference import simulation_wrapper_sbi, sample_posterior
from cstr_sbi.metrics import summarise_posterior
from cstr_sbi.priors import ALPHA_LOW, ALPHA_HIGH, BETA_LOW, BETA_HIGH, box_uniform_2d
from cstr_sbi.summaries import compute_summary_statistics_batch

jax.config.update("jax_enable_x64", False)

ROOT = Path.cwd().parent.parent if (Path.cwd() / "system_I").exists() else Path.cwd().parent
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parents[1]
DATA = ROOT / "data"; DATA.mkdir(exist_ok=True)
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
FIGS = ROOT / "figures"; FIGS.mkdir(exist_ok=True)

print(f"ROOT={ROOT}")
print("JAX devices:", jax.devices())

prior = box_uniform_2d()
Y0_TRAIN = warm_start_ic(NOMINAL_PARAMS_CL, NOMINAL_INLET_CL, NOMINAL_CTRL)


ROOT=/Users/simo/inso-po-RD/cstr-model-optimisation/research_project
JAX devices: [CpuDevice(id=0)]


## Section 2 — Generate (or load/resume) the shared 10,000-draw matched-protocol training bank

Checkpointed every 200 accepted draws to a `.partial.npz` sibling (this project's established convention, e.g. notebooks 22e-22g) -- an interruption loses at most a couple of minutes of work, not the whole ~90 min generation.


In [2]:
N_BANK = 10_000
BANK_PATH = DATA / f"sysI_ablation_train_bank_n{N_BANK}.npz"
BANK_PARTIAL_PATH = BANK_PATH.with_suffix(".partial.npz")
BANK_SEED = 20260930
CHECKPOINT_EVERY = 200

torch.manual_seed(BANK_SEED)
theta_t_full = prior.sample((N_BANK,))

if BANK_PATH.exists():
    _d = np.load(BANK_PATH)
    theta_bank, x_bank = _d["theta"], _d["x"]
    print(f"Loaded complete bank: {BANK_PATH.name}  theta={theta_bank.shape}  x={x_bank.shape}")
else:
    if BANK_PARTIAL_PATH.exists():
        _d = np.load(BANK_PARTIAL_PATH)
        x_list = [torch.as_tensor(row[None, :], dtype=torch.float32) for row in _d["x"]]
        n_done = len(x_list)
        print(f"Resuming from checkpoint: {n_done}/{N_BANK} draws already done.")
    else:
        x_list = []
        n_done = 0
        print(f"Generating {N_BANK} matched-protocol training draws from scratch "
              "(this is the expensive step -- checkpointed every "
              f"{CHECKPOINT_EVERY} draws)...")

    t0 = time.perf_counter()
    for i in range(n_done, N_BANK):
        x_i = simulation_wrapper_sbi(
            theta_t_full[i:i + 1], inlet=NOMINAL_INLET_CL, ctrl=NOMINAL_CTRL, y0=Y0_TRAIN,
            seed=BANK_SEED + i, scenario_specific_warm_start=True)
        x_list.append(x_i)
        done = i + 1
        if done % CHECKPOINT_EVERY == 0 or done == N_BANK:
            elapsed = time.perf_counter() - t0
            rate = elapsed / (done - n_done) if done > n_done else float("nan")
            print(f"  {done}/{N_BANK}  ({elapsed:.0f}s this session, {rate:.3f}s/draw, "
                  f"ETA {rate*(N_BANK-done):.0f}s)")
            x_ckpt = torch.cat(x_list).numpy()
            theta_ckpt = theta_t_full[:done].numpy()
            np.savez(BANK_PARTIAL_PATH, theta=theta_ckpt, x=x_ckpt, seed=BANK_SEED)

    x_bank = torch.cat(x_list).numpy()
    theta_bank = theta_t_full.numpy()
    np.savez(BANK_PATH, theta=theta_bank, x=x_bank, seed=BANK_SEED)
    if BANK_PARTIAL_PATH.exists():
        BANK_PARTIAL_PATH.unlink()
    print(f"Done. Saved {BANK_PATH.name}")


Generating 10000 matched-protocol training draws from scratch (this is the expensive step -- checkpointed every 200 draws)...


  200/10000  (107s this session, 0.536s/draw, ETA 5254s)


  400/10000  (209s this session, 0.524s/draw, ETA 5027s)


  600/10000  (313s this session, 0.522s/draw, ETA 4908s)


  800/10000  (401s this session, 0.502s/draw, ETA 4617s)


  1000/10000  (496s this session, 0.496s/draw, ETA 4464s)


... (41 outputs left out)

  9400/10000  (4641s this session, 0.494s/draw, ETA 296s)


  9600/10000  (4736s this session, 0.493s/draw, ETA 197s)


  9800/10000  (4830s this session, 0.493s/draw, ETA 99s)


  10000/10000  (4916s this session, 0.492s/draw, ETA 0s)
Done. Saved sysI_ablation_train_bank_n10000.npz


## Section 3 — Generate (or load/resume) the shared held-out SBC evaluation set

A reserved seed range, distinct from the training bank, never pooled into training -- the one honest calibration check every ablation config is scored against. Also checkpointed.


In [3]:
N_HELDOUT = 300
HELDOUT_PATH = RESULTS / f"sysI_ablation_sbc_heldout_n{N_HELDOUT}.npz"
HELDOUT_PARTIAL_PATH = HELDOUT_PATH.with_suffix(".partial.npz")
HELDOUT_SEED = 999_000
HELDOUT_CHECKPOINT_EVERY = 100

torch.manual_seed(HELDOUT_SEED)
theta_ho_full = prior.sample((N_HELDOUT,))

if HELDOUT_PATH.exists():
    _d = np.load(HELDOUT_PATH)
    theta_heldout, x_heldout = _d["theta"], _d["x"]
    print(f"Loaded complete held-out set: {HELDOUT_PATH.name}")
else:
    if HELDOUT_PARTIAL_PATH.exists():
        _d = np.load(HELDOUT_PARTIAL_PATH)
        x_list = [torch.as_tensor(row[None, :], dtype=torch.float32) for row in _d["x"]]
        n_done = len(x_list)
        print(f"Resuming held-out set from checkpoint: {n_done}/{N_HELDOUT} draws already done.")
    else:
        x_list = []
        n_done = 0
        print(f"Generating {N_HELDOUT} held-out matched-protocol SBC draws from scratch...")

    t0 = time.perf_counter()
    for i in range(n_done, N_HELDOUT):
        x_i = simulation_wrapper_sbi(
            theta_ho_full[i:i + 1], inlet=NOMINAL_INLET_CL, ctrl=NOMINAL_CTRL, y0=Y0_TRAIN,
            seed=HELDOUT_SEED + i, scenario_specific_warm_start=True)
        x_list.append(x_i)
        done = i + 1
        if done % HELDOUT_CHECKPOINT_EVERY == 0 or done == N_HELDOUT:
            elapsed = time.perf_counter() - t0
            print(f"  {done}/{N_HELDOUT}  ({elapsed:.0f}s this session)")
            x_ckpt = torch.cat(x_list).numpy()
            theta_ckpt = theta_ho_full[:done].numpy()
            np.savez(HELDOUT_PARTIAL_PATH, theta=theta_ckpt, x=x_ckpt, seed=HELDOUT_SEED)

    x_heldout = torch.cat(x_list).numpy()
    theta_heldout = theta_ho_full.numpy()
    np.savez(HELDOUT_PATH, theta=theta_heldout, x=x_heldout, seed=HELDOUT_SEED)
    if HELDOUT_PARTIAL_PATH.exists():
        HELDOUT_PARTIAL_PATH.unlink()
    print(f"Done. Saved {HELDOUT_PATH.name}")


Generating 300 held-out matched-protocol SBC draws from scratch...


  100/300  (40s this session)


  200/300  (86s this session)


  300/300  (120s this session)
Done. Saved sysI_ablation_sbc_heldout_n300.npz


## Section 4 — Lean train-from-bank and SBC-evaluation helpers

Trains directly on pre-computed `(theta, x)` pairs via the raw `sbi.inference.SNPE` API, bypassing `train_sbi_posterior`'s own fresh-simulation loop -- this is what makes reusing the shared bank possible.


In [4]:
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
from sbi.diagnostics import run_sbc, check_sbc


def train_from_bank(theta_np, x_np, density_estimator="nsf", hidden_features=128,
                     num_transforms=5, seed=0, max_num_epochs=200, training_batch_size=256):
    torch.manual_seed(seed)
    net_builder = posterior_nn(
        model=density_estimator, hidden_features=hidden_features,
        num_transforms=num_transforms, z_score_x="independent",
    )
    inference = SNPE(prior=prior, density_estimator=net_builder, show_progress_bars=False)
    theta_t = torch.as_tensor(theta_np, dtype=torch.float32)
    x_t = torch.as_tensor(x_np, dtype=torch.float32)
    t0 = time.perf_counter()
    inference.append_simulations(theta_t, x_t)
    density_est = inference.train(
        training_batch_size=training_batch_size, max_num_epochs=max_num_epochs,
        show_train_summary=False,
    )
    wall_time = time.perf_counter() - t0
    posterior = inference.build_posterior(density_est)
    return posterior, wall_time


def evaluate_sbc(posterior, theta_heldout_np, x_heldout_np, num_posterior_samples=500):
    theta_t = torch.as_tensor(theta_heldout_np, dtype=torch.float32)
    x_t = torch.as_tensor(x_heldout_np, dtype=torch.float32)
    ranks, dap_samples = run_sbc(theta_t, x_t, posterior,
                                  num_posterior_samples=num_posterior_samples, reduce_fns="marginals")
    checks = check_sbc(ranks, theta_t, dap_samples, num_posterior_samples=num_posterior_samples)
    return {
        "ks_p_alpha": float(checks["ks_pvals"][0]),
        "ks_p_beta": float(checks["ks_pvals"][1]),
        "c2st_alpha": float(checks["c2st_ranks"][0]),
        "c2st_beta": float(checks["c2st_ranks"][1]),
    }


def evaluate_scenario_points(posterior):
    # Sc1 (healthy) and Sc2 (jacket fouling) posterior mean/coverage, matching
    # this project's established single-point diagnostic style (S3).
    obs_sc1 = np.asarray(simulation_wrapper_sbi(
        torch.tensor([[1.0, 1.0]]), inlet=NOMINAL_INLET_CL, ctrl=NOMINAL_CTRL, y0=Y0_TRAIN,
        seed=424242, scenario_specific_warm_start=True))[0]
    obs_sc2 = np.asarray(simulation_wrapper_sbi(
        torch.tensor([[1.0, 0.7]]), inlet=NOMINAL_INLET_CL, ctrl=NOMINAL_CTRL, y0=Y0_TRAIN,
        seed=434343, scenario_specific_warm_start=True))[0]
    s1 = sample_posterior(posterior, obs_sc1, n_samples=5000)
    s2 = sample_posterior(posterior, obs_sc2, n_samples=5000)
    m1 = summarise_posterior(s1, np.array([1.0, 1.0]))
    m2 = summarise_posterior(s2, np.array([1.0, 0.7]))
    return {
        "sc1_alpha_mean": m1["alpha"]["mean"], "sc1_beta_mean": m1["beta"]["mean"],
        "sc2_beta_mean": m2["beta"]["mean"], "sc2_beta_cov90": m2["beta"]["coverage_90"],
    }

print("Helpers defined.")


Helpers defined.


## Section 5 — Run the ablation grid

9 configs total (1 shared production reference + 4 budget points + 3 architecture points + 1 estimator-family point). Each trains on the shared bank (or a prefix of it) and is scored against the shared held-out set.


In [5]:
CONFIGS = [
    # (name, axis, n_sim, density_estimator, hidden_features, num_transforms)
    ("production (n=10000, NSF 128/5)", "reference", 10_000, "nsf", 128, 5),
    ("budget: n=500",   "budget", 500,   "nsf", 128, 5),
    ("budget: n=1000",  "budget", 1_000, "nsf", 128, 5),
    ("budget: n=2500",  "budget", 2_500, "nsf", 128, 5),
    ("budget: n=5000",  "budget", 5_000, "nsf", 128, 5),
    ("arch: 64 hidden / 3 transforms",  "architecture", 10_000, "nsf", 64, 3),
    ("arch: 256 hidden / 5 transforms", "architecture", 10_000, "nsf", 256, 5),
    ("arch: 128 hidden / 8 transforms", "architecture", 10_000, "nsf", 128, 8),
    ("estimator: MAF (128/5)", "estimator", 10_000, "maf", 128, 5),
]

RESULTS_CSV_PATH = RESULTS / "sysI_ablation_study.csv"

# Resumable: skip configs whose name is already a row in a partial/complete CSV on disk.
if RESULTS_CSV_PATH.exists():
    rows = pd.read_csv(RESULTS_CSV_PATH).to_dict("records")
    done_names = {r["name"] for r in rows}
    print(f"Resuming: {len(rows)}/{len(CONFIGS)} configs already completed: {sorted(done_names)}")
else:
    rows = []
    done_names = set()

t_grid0 = time.perf_counter()
for cfg_idx, (name, axis, n_sim, dens, hf, nt) in enumerate(CONFIGS):
    if name in done_names:
        print(f"\n=== [{cfg_idx+1}/{len(CONFIGS)}] {name} -- SKIPPED (already done) ===")
        continue
    print(f"\n=== [{cfg_idx+1}/{len(CONFIGS)}] {name} ===")
    theta_sub = theta_bank[:n_sim]
    x_sub = x_bank[:n_sim]
    posterior, wall_time = train_from_bank(
        theta_sub, x_sub, density_estimator=dens, hidden_features=hf, num_transforms=nt, seed=0,
    )
    print(f"  trained in {wall_time:.1f}s")
    sbc = evaluate_sbc(posterior, theta_heldout, x_heldout)
    scen = evaluate_scenario_points(posterior)
    row = dict(name=name, axis=axis, n_sim=n_sim, density_estimator=dens,
               hidden_features=hf, num_transforms=nt, train_wall_time_s=wall_time,
               **sbc, **scen)
    rows.append(row)
    # Checkpoint after every single config -- each one costs real time (data is cached,
    # but training + SBC evaluation on 300 held-out points is not instant).
    pd.DataFrame(rows).to_csv(RESULTS_CSV_PATH, index=False)
    print(f"  KS p: alpha={sbc['ks_p_alpha']:.4f}  beta={sbc['ks_p_beta']:.4f}   "
          f"Sc2 beta_hat={scen['sc2_beta_mean']:.3f}   (checkpointed to {RESULTS_CSV_PATH.name})")

print(f"\nTotal ablation grid time this session: {time.perf_counter()-t_grid0:.0f}s")
results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_CSV_PATH, index=False)
print(f"Saved {RESULTS_CSV_PATH}")
print(results_df.to_string(index=False))



=== [1/9] production (n=10000, NSF 128/5) ===


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/inference/trainers/npe/npe_base.py:196: UserWarning: Data has extreme outliers in dimension(s) [2] (beyond 10.0x IQR from quartiles). This may cause precision loss during z-scoring, where distinct values become indistinguishable. Consider removing outliers from your data or setting `z_score_x='none'` (though this may affect training).
  warn_if_invalid_for_zscoring(x)


 Neural network successfully converged after 119 epochs.  trained in 84.3s


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/utils/diagnostics_utils.py:45: UserWarning: Capping max_sampling_batch_size from 10000 to 333 to avoid excessive memory usage.
  posterior_samples = posterior.sample_batched(


Drawing 500 samples for 300 observations:   0%|          | 0/500 [00:00<?, ?it/s]

/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/nflows/transforms/lu.py:80: UserWarning: torch.triangular_solve is deprecated in favor of torch.linalg.solve_triangularand will be removed in a future PyTorch release.
torch.linalg.solve_triangular has its arguments reversed and does not return a copy of one of the inputs.
X = torch.triangular_solve(B, A).solution
should be replaced with
X = torch.linalg.solve_triangular(A, B). (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/BatchLinearAlgebra.cpp:2279.)
  outputs, _ = torch.triangular_solve(


... (46 outputs left out)

/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/utils/diagnostics_utils.py:45: UserWarning: Capping max_sampling_batch_size from 10000 to 333 to avoid excessive memory usage.
  posterior_samples = posterior.sample_batched(


Drawing 500 samples for 300 observations:   0%|          | 0/500 [00:00<?, ?it/s]

Calculating ranks for 300 SBC samples:   0%|          | 0/300 [00:00<?, ?it/s]

  KS p: alpha=0.0000  beta=0.0558   Sc2 beta_hat=0.690   (checkpointed to sysI_ablation_study.csv)

Total ablation grid time this session: 455s
Saved /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/results/sysI_ablation_study.csv
                           name         axis  n_sim density_estimator  hidden_features  num_transforms  train_wall_time_s   ks_p_alpha  ks_p_beta  c2st_alpha  c2st_beta  sc1_alpha_mean  sc1_beta_mean  sc2_beta_mean  sc2_beta_cov90
production (n=10000, NSF 128/5)    reference  10000               nsf              128               5          84.265737 1.258048e-01   0.000820    0.550000   0.556667        1.000197       1.020713       0.704911            True
                  budget: n=500       budget    500               nsf              128               5           3.396836 1.287990e-03   0.095226    0.531667   0.526667        1.002785       1.022263       0.727818            True
                 budget: n=1000       budget   1000          

## Section 6 — Figure: budget and architecture trends


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

budget_df = results_df[results_df["axis"].isin(["budget", "reference"])].sort_values("n_sim")
ax = axes[0]
ax.plot(budget_df["n_sim"], budget_df["ks_p_alpha"], "o-", label="KS p (alpha)")
ax.plot(budget_df["n_sim"], budget_df["ks_p_beta"], "s-", label="KS p (beta)")
ax.axhline(0.05, color="r", ls="--", lw=1, label="p=0.05")
ax.set_xscale("log")
ax.set_xlabel("n_simulations"); ax.set_ylabel("SBC KS p-value")
ax.set_title("Calibration vs. training budget")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(budget_df["n_sim"], budget_df["sc2_beta_mean"], "o-", color="tomato")
ax.axhline(0.70, color="k", ls="--", lw=1, label="true beta=0.70")
ax.set_xscale("log")
ax.set_xlabel("n_simulations"); ax.set_ylabel("Sc2 beta posterior mean")
ax.set_title("Point accuracy vs. training budget")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

arch_df = results_df[results_df["axis"].isin(["architecture", "estimator", "reference"])]
labels = arch_df["name"].str.replace("arch: ", "").str.replace("estimator: ", "").str.replace("production.*", "production", regex=True)
x_pos = np.arange(len(arch_df))
ax = axes[2]
width = 0.35
ax.bar(x_pos - width/2, arch_df["ks_p_alpha"], width, label="KS p (alpha)")
ax.bar(x_pos + width/2, arch_df["ks_p_beta"], width, label="KS p (beta)")
ax.axhline(0.05, color="r", ls="--", lw=1)
ax.set_xticks(x_pos); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("SBC KS p-value")
ax.set_title("Calibration vs. architecture / estimator")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")

fig.suptitle("System I SBI hyperparameter ablation (shared bank + shared held-out SBC set)")
fig_path = FIGS / "18_ablation_study.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {fig_path}")


Saved: /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/figures/18_ablation_study.png


## Section 7 — Summary


In [7]:
print("=== Ablation study complete ===")
print(results_df[["name", "n_sim", "density_estimator", "hidden_features", "num_transforms",
                   "train_wall_time_s", "ks_p_alpha", "ks_p_beta", "sc2_beta_mean"]].to_string(index=False))
print()
print(f"Results table: {RESULTS / 'sysI_ablation_study.csv'}")
print(f"Figure: {FIGS / '18_ablation_study.png'}")
print()
print("Next: add a new S3a section to docs/Manuscript/supporting_information.tex")
print("presenting this table and figure.")


=== Ablation study complete ===
                           name  n_sim density_estimator  hidden_features  num_transforms  train_wall_time_s   ks_p_alpha  ks_p_beta  sc2_beta_mean
production (n=10000, NSF 128/5)  10000               nsf              128               5          84.265737 1.258048e-01   0.000820       0.704911
                  budget: n=500    500               nsf              128               5           3.396836 1.287990e-03   0.095226       0.727818
                 budget: n=1000   1000               nsf              128               5           7.344429 1.077105e-03   0.059303       0.701692
                 budget: n=2500   2500               nsf              128               5          13.635763 1.933491e-02   0.075472       0.692480
                 budget: n=5000   5000               nsf              128               5          43.524122 4.266737e-06   0.006320       0.698997
 arch: 64 hidden / 3 transforms  10000               nsf               64       